# Vyuha P16 — head-to-head on PIArena (vs the cited attackers)

Slots **Vyuha** into **PIArena** (Geng et al., ACL 2026 — arXiv:2604.08499) as a Filter-type defense
and attacks it with the *same* static + search suite **PISmith** (Yin et al., COLM 2026 —
arXiv:2603.13026) benchmarks on: **Direct, Combined, Ignore, Completion, Character** (static) and
**PAIR / TAP** (search). Reports Vyuha's **ASR@1** next to the `none` baseline, so it can be placed
against PIArena's published defenses (PIGuard, PromptGuard, DataSentinel, …).

**Honest scope.** PISmith's *RL* attacker trains an attack LLM (GRPO, ~4 GPUs, no released
checkpoint) — out of free-compute scope; it is the cited upper bound. This notebook runs the
free-compute-feasible part: static attacks on a single GPU (+ light search).

**Requirements.** Kaggle/Colab **GPU** session, **Internet ON**, and a **HF token** (some Vyuha
training datasets are gated — accept their terms first). The adapter lives in the Vyuha repo at
`integrations/piarena/`; this notebook only orchestrates clone → install → register → run → collect.

## 1 · Setup — clone Vyuha + PIArena, install, register the `vyuha` defense

In [ ]:
import os, sys, glob, subprocess, shutil, textwrap

VYUHA_URL = "https://github.com/g25ait2149/vyuha.git"
PIARENA_URL = "https://github.com/sleeepeer/PIArena.git"
WORK = "/kaggle/working"
VYUHA_DIR = f"{WORK}/vyuha_src"
PIARENA_DIR = f"{WORK}/PIArena"

def sh(cmd, **kw):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, **kw)

# --- Vyuha (adapter + vyuha package + eval) ---
if os.path.isdir(f"{VYUHA_DIR}/.git"):
    sh(f'git -C "{VYUHA_DIR}" pull --ff-only')
else:
    sh(f'git clone --depth 1 {VYUHA_URL} "{VYUHA_DIR}"')
hits = glob.glob(VYUHA_DIR + "/**/vyuha/__init__.py", recursive=True)
VYUHA_ROOT = os.path.dirname(os.path.dirname(hits[0])) if hits else VYUHA_DIR
print("vyuha repo root:", VYUHA_ROOT)

# --- PIArena (the platform) ---
if os.path.isdir(f"{PIARENA_DIR}/.git"):
    sh(f'git -C "{PIARENA_DIR}" pull --ff-only')
else:
    sh(f'git clone --depth 1 {PIARENA_URL} "{PIARENA_DIR}"')

# --- install ONLY the light deps main.py's import chain needs. PIArena's requirements.txt pins
# vllm + fschat[model_worker,webui], which fail to build on Kaggle and are NOT needed for the
# transformers backend on the judge-free path. piarena imports from cwd, so no editable install.
sh('pip -q install openai google-genai google-generativeai anthropic rich fuzzywuzzy levenshtein '
   'rouge jieba scipy scikit-learn 2>&1 | tail -3')

# --- drop the Vyuha adapter into piarena/defenses/ and register it ---
adapter_src = f"{VYUHA_ROOT}/integrations/piarena"
dest = f"{PIARENA_DIR}/piarena/defenses"
for f in ("_vyuha_core.py", "defense_vyuha.py"):
    shutil.copy(f"{adapter_src}/{f}", f"{dest}/{f}")
    print("copied", f, "->", dest)
init = f"{dest}/__init__.py"
reg = "from .defense_vyuha import VyuhaDefense  # noqa: F401"
txt = open(init).read()
if reg not in txt:
    open(init, "a").write("\n" + reg + "\n")
    print("registered VyuhaDefense in", init)
else:
    print("VyuhaDefense already registered")

# --- so PIArena's process can import `vyuha` and `eval` from the Vyuha repo ---
os.environ["PYTHONPATH"] = VYUHA_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
print("PYTHONPATH ->", os.environ["PYTHONPATH"][:120], "...")
print("\nSetup done.")

## 2 · Hugging Face login (gated datasets)
Vyuha's L1 detector is fit on assembled corpora, some **gated** (AdvBench, HarmBench, WildGuardMix).
Accept their terms on HF, then set your token below (or add a Kaggle Secret `HF_TOKEN`).

In [ ]:
import os
# Load HF token from a Kaggle Secret (or paste). Setting the env var is all PIArena/datasets need;
# no CLI login required (huggingface-cli is deprecated). PIArena datasets are public (MIT); HF_TOKEN
# mainly helps the Vyuha adapter fit L1 on its full corpus (it falls back to a bundled seed set if not).
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secret')
except Exception:
    os.environ.setdefault('HF_TOKEN', '')   # <-- or paste your token here
    print('HF_TOKEN set:', bool(os.environ['HF_TOKEN']))
os.environ.setdefault('HUGGINGFACE_HUB_TOKEN', os.environ.get('HF_TOKEN', ''))


## 3 · Config — dataset, attacks, defenses to compare

In [ ]:
# JUDGE-FREE setup: knowledge_corruption datasets use substring_match for BOTH ASR and utility
# (PIArena main.py), so NO judge LLM / API key is needed - only the Qwen-4B backend on the T4.
DATASET = 'nq_rag_knowledge_corruption'   # or hotpotqa_rag_knowledge_corruption / msmarco_rag_knowledge_corruption
STATIC_ATTACKS = ['direct', 'combined', 'ignore', 'completion', 'character']   # no attacker LLM needed
SEARCH_ATTACKS = ['pair']                 # heavier; needs an attacker LLM (set RUN_SEARCH=True)
DEFENSES = ['none', 'vyuha']              # baseline vs Vyuha; add 'piguard','promptguard' for live baselines
BACKEND_LLM = 'Qwen/Qwen3-4B-Instruct-2507'   # target model; fits the T4x2
ATTACKER_LLM = 'Qwen/Qwen3-4B-Instruct-2507'
RUN_SEARCH = False
NUM_SAMPLES = 5
NAME = 'vyuha_h2h'                         # results land in results/evaluation_results/<NAME>/
import os, json
VYUHA_USE_GUARD = False                    # True -> also load the L2 guard (Qwen3Guard-0.6B); fits alongside 4B on T4x2
if VYUHA_USE_GUARD:
    os.environ['VYUHA_DEFENSE_CONFIG'] = json.dumps({'use_guard': True, 'guard_preset': 'qwen3guard'})
print('config: dataset', DATASET, '| judge-free (substring_match) | defenses', DEFENSES)


## 3b · Smoke test — is `vyuha` registered and does PIArena run?
Runs two checks with **full output printed inline** (no log files needed). If either fails, the error here tells us exactly what to fix before the batch.

In [ ]:
import os, subprocess
env = {**os.environ}
# (1) does the Vyuha defense import + register inside PIArena's env?
chk = subprocess.run(['python','-c',
    'from piarena.defenses import get_defense; d=get_defense("vyuha"); print("OK registered:", d)'],
    cwd=PIARENA_DIR, env=env, capture_output=True, text=True)
print('--- defense import check ---'); print(chk.stdout or ''); print(chk.stderr or ''); print('exit', chk.returncode)
# (2) minimal end-to-end on the judge-free dataset (attack=none, defense=none) - full output inline
smoke = subprocess.run(['python','main.py','--dataset',DATASET,'--attack','none','--defense','none',
                        '--backend_llm',BACKEND_LLM,'--name',NAME+'_smoke'],
                       cwd=PIARENA_DIR, env=env, capture_output=True, text=True)
print('\n--- main.py smoke (attack=none, defense=none) ---')
print((smoke.stdout or '')[-4000:]); print('--- stderr ---'); print((smoke.stderr or '')[-2500:]); print('exit', smoke.returncode)


## 4 · Run the static attacks
Each run: `python main.py --dataset <d> --attack <a> --defense <def>`. We capture stdout and try to
parse an ASR value; the full log is saved to `/kaggle/working/piarena_logs/` regardless.

In [ ]:
import os, subprocess, itertools, pathlib
LOGDIR = '/kaggle/working/piarena_logs'; os.makedirs(LOGDIR, exist_ok=True)
env = {**os.environ}

def run_one(entry, attack, defense, search=False):
    script = 'main_search.py' if search else 'main.py'
    cmd = ['python', script, '--dataset', DATASET, '--attack', attack, '--defense', defense,
           '--backend_llm', BACKEND_LLM, '--name', NAME]
    if search:
        cmd += ['--attacker_llm', ATTACKER_LLM, '--num_samples', str(NUM_SAMPLES)]
    print('\n>>>', ' '.join(cmd))
    p = subprocess.run(cmd, cwd=PIARENA_DIR, env=env, capture_output=True, text=True)
    out = (p.stdout or '') + '\n' + (p.stderr or '')
    (pathlib.Path(LOGDIR)/f'{entry}.log').write_text(out)
    print('   exit', p.returncode)
    if p.returncode != 0:
        print('   --- output tail (diagnose) ---'); print('\n'.join(out.strip().splitlines()[-25:]))
    return {'attack': attack, 'defense': defense, 'search': search, 'exit': p.returncode}

results = []
for atk, dfn in itertools.product(STATIC_ATTACKS, DEFENSES):
    results.append(run_one(f'{atk}__{dfn}', atk, dfn))
print('\nstatic runs done:', len(results))


## 5 · (Optional) search-based attacks — PAIR / TAP

In [ ]:
import itertools
if RUN_SEARCH:
    for atk, dfn in itertools.product(SEARCH_ATTACKS, DEFENSES):
        results.append(run_one(f"{atk}__{dfn}", atk, dfn, search=True))
    print("search runs complete.")
else:
    print("RUN_SEARCH=False — skipping PAIR/TAP. Set True in the config cell to include them.")

## 6 · Collect results — Vyuha vs baseline (+ any live defenses)

In [ ]:
import glob, json, pathlib, numpy as np, pandas as pd
base = f'{PIARENA_DIR}/results/evaluation_results/{NAME}'
rows = []
for f in sorted(glob.glob(base + '/*.json')):        # eval results only (tmp_attack_results is a subdir)
    stem = pathlib.Path(f).stem                       # {dataset}-{llm}-{attack}-{defense}-{seed}
    try:
        d = json.load(open(f))
    except Exception as e:
        print('skip', f, e); continue
    vals = list(d.values())
    if not vals: continue
    parts = stem.split('-')                           # attack/defense/seed have no dashes -> take from the end
    seed, defense, attack = parts[-1], parts[-2], parts[-3]
    rows.append({'attack': attack, 'defense': defense, 'n': len(vals),
                 'ASR': round(float(np.nanmean([v.get('asr', np.nan) for v in vals])), 3),
                 'utility': round(float(np.nanmean([v.get('utility', np.nan) for v in vals])), 3)})
df = pd.DataFrame(rows)
if df.empty:
    print('No result files in', base)
    print('-> open the run logs in /kaggle/working/piarena_logs/*.log to see what errored.')
else:
    print('=== ASR@1 : Vyuha vs baseline on', DATASET, '(judge-free substring_match) ===')
    print(df.pivot_table(index='attack', columns='defense', values='ASR').to_string())
    print('\n=== utility (task accuracy, attack=none rows) ===')
    print(df.pivot_table(index='attack', columns='defense', values='utility').to_string())
    df.to_csv('/kaggle/working/piarena_vyuha_results.csv', index=False)
    print('\nsaved -> /kaggle/working/piarena_vyuha_results.csv   (send me this table)')


## 7 · Interpretation & what to send back
- **Comparison to publish:** Vyuha's **ASR@1 per attack** vs `none`, placed next to PIArena's published
  defenses. In their paper (Qwen3-4B target) the filter defenses sit around: PIGuard 0.82, PromptGuard
  0.89, DataSentinel 0.52 (PISmith ASR@1). Static/search baselines are far weaker (Direct 0.04,
  Combined 0.07, PAIR 0.16, TAP 0.11) — that is the band this run measures Vyuha against.
- **Utility:** run `--attack none --defense vyuha` to get task accuracy without attack (the utility
  column); Vyuha passes benign contexts through unchanged, so utility should stay near the `none`
  baseline.
- **Honesty rule:** report the measured ASR as-is. If Vyuha underperforms a published filter, that is a
  finding for the results + Limitations, not something to bury.
- **Send me:** `piarena_vyuha_results.csv` (or the printed table) and I'll fold it into the paper's
  results table and §7.4, alongside the pairwise + genetic adaptive numbers from P14.

*The RL attacker (PISmith) itself needs a ~4-GPU training rig and is cited as the stronger upper bound;
this notebook covers the free-compute-feasible static + search head-to-head.*